# Engine: CorpusProcessor — feed any text archive to ValaQuenta

**File:** `ValaQuenta/corpus.py`
**Wiki:** [wiki/corpus.md](../../wiki/corpus.md)

Reads files, splits into passages, sets the semantic domain **from context**,
processes every word, records every prime, builds the lexicon.

**Status: untested at scale.** No binary corpus has been loaded — this is
recorded as an open item in the wiki index and it is still open. This notebook
builds a small archive in a temporary directory so the pipeline can be exercised
end to end, but a handful of files is not a corpus and this is not evidence that
the engine works at archive scale.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
import math, cmath
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('python', sys.version.split()[0])

In [ ]:
from ValaQuenta import CorpusProcessor, Understand, Lexicon
import tempfile, os as _os

work = tempfile.mkdtemp(prefix='valaquenta_corpus_')
passages = {
    'trees.txt': ('A tree is a woody perennial plant. The tree has a trunk. '
                  'In graph theory a tree is an acyclic connected graph.'),
    'primes.txt': ('A prime has no divisors but one and itself. '
                   'The primes are enumerated by the hyperbola xp = E.'),
    'zeros.txt': ('The Riemann zeros lie on the critical line. '
                  'The real part is one half.'),
}
for name, text in passages.items():
    with open(_os.path.join(work, name), 'w') as f:
        f.write(text)
print('archive at', work)
print('files:', sorted(_os.listdir(work)))

In [ ]:
cp = CorpusProcessor()
print('CorpusProcessor:', cp)
print('public methods:', [m for m in dir(cp) if not m.startswith('_')])

The cell below walks the archive with whatever entry point the class
exposes. It is written defensively on purpose: this engine is the least
exercised in the repo, and a notebook that pretends otherwise would be
misleading.

In [ ]:
import traceback

entry = None
for name in ['process_directory', 'process_dir', 'process_archive',
             'process_file', 'process', 'run']:
    if hasattr(cp, name):
        entry = name
        break
print('entry point found:', entry)

if entry:
    fn = getattr(cp, entry)
    try:
        if 'file' in entry:
            out = fn(_os.path.join(work, 'trees.txt'))
        else:
            out = fn(work)
        print('OK ->', str(out)[:500])
    except Exception:
        print('FAILED -- kept, not hidden:')
        traceback.print_exc(limit=3)
else:
    print('no directory entry point exposed; falling back to per-word processing')

## Fallback: the pipeline done by hand

In [ ]:
U = Understand(tau=1.0)
lex = Lexicon()

for name, text in passages.items():
    domain = U.describe(text)
    print(f'{name}: domain span = {domain.span!r}')
    for word in text.replace('.', ' ').split():
        w = U.process(word, domain=domain)
        lex.record(w.gamma, word.lower(), domain=name)

print()
print(f'distinct gammas recorded: {len(lex.known_gammas())}')
for g in lex.known_gammas()[:8]:
    print(f'  {g!r:>22}  faces={lex.face_count(g):>3}  best={lex.best_face(g)!r}')

In [ ]:
import shutil
shutil.rmtree(work, ignore_errors=True)
print('cleaned up', work)